# Analysis of experiment #6: Pricing methods

## Read data

In [39]:
# Read csv

import pandas as pd
import numpy as np

df = pd.read_csv("stats6.csv")

# Rename time limit states
df.loc[df.time == 900, "state"] = "TIMELIMIT"
df["timeSolved"] = df.apply(lambda row: float("nan") if row.state == "TIMELIMIT" else row.time, axis=1)

# Add column with edge probability
df["p"] = df.instance.apply(lambda s: float(s.split("_")[2][1:]))

df

,instance,solver,run,nvertices,nedges,n,m,nvars,ncons,state,...,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,timeSolved,p
0,r_n130_p0.25_nA26_nB13_i1,byp_p4,0,130,2181,26,13,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,246.795,0.25
1,r_n170_p0.75_nA34_nB17_i2,byp_p4,0,170,10813,34,17,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,352.118,0.75
2,r_n170_p0.75_nA34_nB17_i0,byp_p4,0,170,10814,34,17,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,415.416,0.75
3,r_n170_p0.75_nA17_nB34_i4,byp_p4,0,170,10756,17,34,-1,-1,OPTIMAL,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,114.015,0.75
4,r_n150_p0.5_nA15_nB30_i4,byp_p4,0,150,5679,15,30,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,120.331,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,r_n170_p0.75_nA17_nB17_i3,byp_p2,0,170,10751,17,17,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,158.637,0.75
296,r_n150_p0.5_nA15_nB30_i0,byp_p2,0,150,5548,15,30,-1,-1,OPTIMAL,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,150.003,0.50
297,r_n150_p0.5_nA15_nB15_i2,byp_p2,0,150,5568,15,15,-1,-1,OPTIMAL,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,273.051,0.50
298,r_n150_p0.5_nA30_nB15_i3,byp_p2,0,150,5675,30,15,-1,-1,TIMELIMIT,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.50


In [40]:
df.columns

Index(['instance', 'solver', 'run', 'nvertices', 'nedges', 'n', 'm', 'nvars',
       'ncons', 'state', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap',
       'ninfeas', 'ninfeasPrepro', 'ninfeasCheck', 'ninfeasAux', 'nint',
       'ngcp', 'gcpTime', 'nsol', 'nsolHeur', 'nsolLR', 'ntrivial', 'rootlb',
       'rootub', 'rootHeurTime', 'rootFeasTime', 'rootNCalls',
       'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1',
       'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool',
       'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact',
       'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1',
       'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime',
       'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls',
       'otherNodesNCallsPool', 'otherNodesNCallsHeur', 'otherNodesNCallsMwis1',
       'otherNodesNCallsMwis2', 'otherNodesNCallsExact', 'otherNodesNCols',
       'otherNodesNColsPool', 'otherNodesNColsHeur', 'other

## Summary

In [43]:
df3 = df.groupby(["solver"]).agg(
    {
        "state": [("solved", lambda x: ((x == "FEASIBLE") | (x == "OPTIMAL")).sum())],
        "time": [("avg", "mean")],
        "rootTimeHeur": [("avg", "mean")],
        "rootTimeMwis1": [("avg", "mean")],
        "rootTimeMwis2": [("avg", "mean")],
        "rootTimeExact": [("avg", "mean")],
        "rootNCols": [("avg", "mean")],
        "rootNColsHeur": [("avg", "mean")],
        "rootNColsMwis1": [("avg", "mean")],
        "rootNColsMwis2": [("avg", "mean")],
        "rootNColsExact": [("avg", "mean")],
    }
).reset_index()
columns = [("Pricing",""), ("#Solved",""), ("Time (s)", "Total"), ("Time (s)","Heur."), ("Time (s)","𝒫-MWSSP"), ("Time (s)","𝒫𝑄-MWSSP"), ("Time (s)","IP"), 
           ("#Columns","Total"), ("#Columns","Heur."), ("#Columns","𝒫-MWSSP"), ("#Columns","𝒫𝑄-MWSSP"), ("#Columns","IP")]
df3.columns=pd.MultiIndex.from_tuples(columns)
df3 = df3.round({
    ("Time (s)", "Total"): 1,
    ("Time (s)","Heur."): 1,
    ("Time (s)","𝒫-MWSSP"): 2,
    ("Time (s)","𝒫𝑄-MWSSP"): 2,
    ("Time (s)","IP"): 1,
    ("#Columns","𝒫-MWSSP"): 1,
    ("#Columns","𝒫𝑄-MWSSP"): 1,
})
df3[("#Columns","Total")] = df3[("#Columns","Total")].round(0).astype(int)
df3[("#Columns","Heur.")] = df3[("#Columns","Heur.")].round(0).astype(int)
df3[("#Columns","IP")] = df3[("#Columns","IP")].round(0).astype(int)
df3 = df3.replace(0, "--")
df3

Pricing #Solved Time (s)                               #Columns        \
                     Total Heur. 𝒫-MWSSP 𝒫𝑄-MWSSP     IP    Total Heur.   
0  byp_p0      27    675.6    --      --       --  628.2      200    --   
1  byp_p1      38    477.9   6.5      --       --  420.6     4790  4766   
2  byp_p2      42    434.4   6.8    0.18       --  374.6     4741  4716   
3  byp_p3      41    457.4   6.6      --      1.4  397.6     4779  4755   
4  byp_p4      37    463.6   6.7    0.17     1.35  405.5     4792  4766   
5  byp_p5      38    454.1   6.7    0.16     1.39  394.5     4800  4774   

                         
  𝒫-MWSSP 𝒫𝑄-MWSSP   IP  
0      --       --  200  
1      --       --   25  
2     4.0       --   21  
3      --      0.6   23  
4     3.7      0.3   22  
5     3.6      0.4   22

In [44]:
df3 = df.groupby(["p","solver"]).agg(
    {
        "state": [("solved", lambda x: ((x == "FEASIBLE") | (x == "OPTIMAL")).sum())],
        "time": [("avg", "mean")],
        "rootTimeHeur": [("avg", "mean")],
        "rootTimeMwis1": [("avg", "mean")],
        "rootTimeMwis2": [("avg", "mean")],
        "rootTimeExact": [("avg", "mean")],
        "rootNCols": [("avg", "mean")],
        "rootNColsHeur": [("avg", "mean")],
        "rootNColsMwis1": [("avg", "mean")],
        "rootNColsMwis2": [("avg", "mean")],
        "rootNColsExact": [("avg", "mean")],
    }
).reset_index()
columns = [("p",""),("Pricing",""), ("#Solved",""), ("Time (s)", "Total"), ("Time (s)","Heur."), ("Time (s)","𝒫-MWSSP"), ("Time (s)","𝒫𝑄-MWSSP"), ("Time (s)","IP"), 
           ("#Columns","Total"), ("#Columns","Heur."), ("#Columns","𝒫-MWSSP"), ("#Columns","𝒫𝑄-MWSSP"), ("#Columns","IP")]
df3.columns=pd.MultiIndex.from_tuples(columns)
df3 = df3.round({
    ("Time (s)", "Total"): 1,
    ("Time (s)","Heur."): 1,
    ("Time (s)","𝒫-MWSSP"): 2,
    ("Time (s)","𝒫𝑄-MWSSP"): 2,
    ("Time (s)","IP"): 1,
    ("#Columns","𝒫-MWSSP"): 1,
    ("#Columns","𝒫𝑄-MWSSP"): 1,
})
df3[("#Columns","Total")] = df3[("#Columns","Total")].round(0).astype(int)
df3[("#Columns","Heur.")] = df3[("#Columns","Heur.")].round(0).astype(int)
df3[("#Columns","IP")] = df3[("#Columns","IP")].round(0).astype(int)
df3 = df3.replace(0, "--")
df3

p Pricing #Solved Time (s)                               #Columns  \
                            Total Heur. 𝒫-MWSSP 𝒫𝑄-MWSSP     IP    Total   
0   0.25  byp_p0       4    807.1    --      --       --  793.3      298   
1   0.25  byp_p1       6    620.4   7.2      --       --  597.3     4182   
2   0.25  byp_p2       8    537.2   7.2     0.4       --  509.2     4087   
3   0.25  byp_p3       6    595.2   7.6      --     5.14  566.6     4130   
4   0.25  byp_p4       6    620.4   7.3    0.37     4.93  592.5     4108   
5   0.25  byp_p5       6    620.6   7.3    0.37     5.04  591.3     4108   
6   0.50  byp_p0      10    669.2    --      --       --  632.7      189   
7   0.50  byp_p1      16    459.1   5.2      --       --  412.3     4313   
8   0.50  byp_p2      17    455.1   5.4    0.14       --  406.1     4237   
9   0.50  byp_p3      18    452.6   5.3      --     0.52  403.6     4248   
10  0.50  byp_p4      14    501.0   5.4    0.14     0.54  455.4     4329   
11  0.50  byp_p5      15    478.6   5.3    0.13     0.55  431.3     4329   
12  0.75  byp_p0      13    616.1    --      --       --  541.3      163   
13  0.75  byp_p1      16    425.5   7.4      --       --  340.6     5572   
14  0.75  byp_p2      17    362.2   7.9    0.11       --  275.8     5572   
15  0.75  byp_p3      17    393.2   7.6      --     0.42  307.0     5634   
16  0.75  byp_p4      17    348.0   7.7     0.1     0.36  262.1     5598   
17  0.75  byp_p5      17    346.5   7.7     0.1      0.4  259.4     5616   

                                
   Heur. 𝒫-MWSSP 𝒫𝑄-MWSSP   IP  
0     --      --       --  298  
1   4143      --       --   39  
2   4053     1.0       --   34  
3   4093      --      0.7   36  
4   4070     0.8      0.5   36  
5   4070     0.8      0.5   36  
6     --      --       --  189  
7   4297      --       --   16  
8   4219     1.3       --   16  
9   4232      --      0.4   16  
10  4310     1.2      0.4   17  
11  4311     1.2      0.5   17  
12    --      --       --  163  
13  5546      --       --   26  
14  5544     8.2       --   20  
15  5609      --      0.6   24  
16  5570     7.7      0.2   20  
17  5589     7.5      0.3   19